# LiteLDM Single Notebook Pipeline

This notebook runs the full pipeline in one place:
1. Environment preflight
2. Data download + preprocessing
3. VAE training
4. Diffusion training
5. Generation + saved outputs

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
from pathlib import Path

import torch
from dotenv import load_dotenv
from huggingface_hub import login

from liteldm.data import (
    Niftify,
    build_slice_loader,
    download_dataset_zip,
    list_nifti_files,
    preprocess_nifti,
    resolve_paths,
)
from liteldm.diffusion import DDPMScheduler
from liteldm.generate import generate_slices, plot_generated, reconstruction_quality_plot, save_generated_tensors
from liteldm.models import DiffusionUNet, VAE, resolve_device
from liteldm.preflight import run_preflight
from liteldm.train import build_latent_loader, encode_dataset_to_latents, train_diffusion, train_vae

print("Imports complete")

In [ ]:
# ===== User Config =====
LOCAL_OVERRIDE = os.environ.get("LOCAL", "./local_storage")
MIN_CONTENT_RATIO = 0.001

ENCODER_BACKBONE = "strainer"  # "conv" or "strainer"
STRAINER_ENCODER_WEIGHTS = None  # e.g. "/path/to/strainer_encoder_weights.pth"

VAE_EPOCHS = 50
DIFF_EPOCHS = 100
NUM_GENERATE = 4

CHECKPOINT_DIR = Path("./checkpoints_notebook")
OUTPUT_DIR = Path("./outputs_notebook")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VAE_CKPT = str(CHECKPOINT_DIR / "vae_ckpt.pt")
DIFF_CKPT = str(CHECKPOINT_DIR / "diffusion_ckpt.pt")

print(f"LOCAL path: {LOCAL_OVERRIDE}")
print(f"Checkpoints: {CHECKPOINT_DIR.resolve()}")
print(f"Outputs: {OUTPUT_DIR.resolve()}")

In [ ]:
# Optional: HuggingFace login from .env
load_dotenv()
hf_token = os.environ.get("huggingface_token")
if hf_token:
    login(hf_token)
    print("HuggingFace login attempted from .env")
else:
    print("No huggingface_token found in environment.")

In [ ]:
# Preflight
run_preflight()

In [ ]:
# Data stage
paths = resolve_paths(LOCAL_OVERRIDE)
zip_path = download_dataset_zip(paths.local)
print(f"Downloaded dataset zip: {zip_path}")

niftify = Niftify(zip_path, paths.dicom_dir, paths.nifti_dir)
niftify.run()

dataset_files = list_nifti_files(paths.nifti_dir)
print(f"NIfTI files found: {len(dataset_files)}")
if not dataset_files:
    raise RuntimeError("No NIfTI files found after conversion")

preprocess_nifti(dataset_files, paths.nifti_dir, paths.processed_dir)
processed_files = list_nifti_files(paths.processed_dir)
print(f"Processed files found: {len(processed_files)}")
if not processed_files:
    raise RuntimeError("No processed NIfTI files found")

In [ ]:
# Build dataset, models, scheduler
dataset, loader = build_slice_loader(paths.processed_dir, min_content_ratio=MIN_CONTENT_RATIO)
sample = next(iter(loader))
print(f"Batch shape: {sample.shape}, dtype: {sample.dtype}, range: [{sample.min():.2f}, {sample.max():.2f}]")

device = resolve_device()
print(f"Using device: {device}")

vae = VAE(
    latent_ch=512,
    encoder_backbone=ENCODER_BACKBONE,
    strainer_hidden_features=256,
    strainer_total_layers=6,
    strainer_shared_encoder_layers=5,
    strainer_num_train_decoders=10,
    strainer_encoder_weights=STRAINER_ENCODER_WEIGHTS,
).to(device)

unet = DiffusionUNet(latent_ch=512).to(device)
ddpm = DDPMScheduler(T=1000, device=str(device))

print("Models initialized")

In [ ]:
# Train VAE
train_vae(vae, loader, device, epochs=VAE_EPOCHS, lr=1e-4, ckpt_path=VAE_CKPT)

In [ ]:
# Train Diffusion
vae.load_state_dict(torch.load(VAE_CKPT, map_location=device))
vae.eval()

latents = encode_dataset_to_latents(vae, loader, device)
print(f"Latent bank: {latents.shape}")
latent_loader = build_latent_loader(latents, batch_size=8)

train_diffusion(unet, ddpm, latent_loader, device, epochs=DIFF_EPOCHS, lr=2e-4, ckpt_path=DIFF_CKPT)

In [ ]:
# Generate and save outputs
vae.load_state_dict(torch.load(VAE_CKPT, map_location=device))
unet.load_state_dict(torch.load(DIFF_CKPT, map_location=device))

generated = generate_slices(unet, vae, ddpm, device, n=NUM_GENERATE, timesteps=1000)

gen_img_path = str(OUTPUT_DIR / "generated_notebook.png")
plot_generated(generated, save_path=gen_img_path, show=False)
print(f"Saved generated image grid: {gen_img_path}")

tensor_path = save_generated_tensors(generated, str(OUTPUT_DIR), prefix="generated_notebook")
print(f"Saved generated tensor batch: {tensor_path}")

In [ ]:
# Reconstruction quality check (saved)
vae.load_state_dict(torch.load(VAE_CKPT, map_location=device))
recon_img_path = str(OUTPUT_DIR / "reconstruction_notebook.png")
reconstruction_quality_plot(vae, loader, device, n=4, save_path=recon_img_path, show=False)
print(f"Saved reconstruction comparison: {recon_img_path}")